# EA2 — Despliegue y gobierno de una infraestructura de datos en la nube

**Big Data (ISD-25)** · Ingeniería de Software y Datos · IU Digital de Antioquia

| | |
|---|---|
| Grupo | 30 |
| Integrantes | Andres Felipe Grisales Vahos · Hanna Jineth Contreras Salinas · Arley Steven Franco Galeano |
| Caso de estudio | Wanderbricks |
| Fecha de entrega | domingo 20 de septiembre |


---
## 1. Contexto y problema

En la EA1 diseñamos la base de datos analítica de Wanderbricks para responder qué combinaciones de dispositivo y canal de origen convierten mejor de navegación a reserva confirmada y pagada. Esa base de datos ya existe y funciona, pero hasta ahora cualquiera con acceso al catálogo puede ver y modificar cualquier capa: no hay diferencia entre quien solo necesita consultar resultados de negocio y quien construye el pipeline.

Esta evidencia responde por el entorno donde esos datos viven: quién puede ver y modificar cada capa (bronce y plata), y cómo se automatiza el procesamiento para que no dependa de que alguien del equipo le dé clic manualmente cada vez. El entorno debe soportar acceso diferenciado por rol (analista de negocio, ingeniero de datos, administrador) y un pipeline que se ejecute solo, de forma programada, sin intervención manual.

---
## 2. Descripción de los datos

El entorno gobierna las mismas 4 tablas de la EA1: bookings (72,247 filas), clickstream (100,000 filas), reviews (99,793 filas) y payments (49,638 filas), organizadas en dos capas dentro del esquema bigdata_grupo30.wanderbricks: bronce (datos crudos, con metadatos de ingesta) y plata (datos limpios y tipados, lista para el análisis).

Aunque en este entorno gratuito los datos de origen no cambian, el pipeline está diseñado para un escenario de actualización diaria: el Job de automatización reingesta y retransforma las 4 tablas cada 24 horas, simulando cómo se vería el proceso con datos reales que sí cambian. Los consumidores previstos son el equipo de marketing (consulta de resultados de conversión) y el equipo de datos (mantenimiento del pipeline).

---
## 3. Decisiones de diseño y justificación
### 3.1 Diagrama de la arquitectura


```
[ FUENTES ]              [ INGESTA ]                [ ALMACENAMIENTO ]           [ PROCESAMIENTO ]              [ CONSUMO ]
samples.wanderbricks  →  Tarea ingesta_bronce     →  Unity Catalog:            →  Tarea transformar_plata    →  Tablas *_plata vía
(dataset de muestra         (código PySpark:            catálogo bigdata_grupo30    (limpieza, dedup, aplanado    GRANT SELECT
 del proveedor)              lee samples.*,               esquema wanderbricks        de structs)                 (analista de negocio,
                             escribe *_bronce)            tablas Delta bronce/plata                                 equipo de marketing)
                             ↑ orquestada por Job          ↑ almacenamiento físico
                             pipeline_wanderbricks          (Delta Lake sobre object
                             (scheduler diario)              storage) gestionado
                                                             por el proveedor

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 Administra el PROVEEDOR (Databricks):  cómputo del clúster/serverless, motor Spark, metastore de Unity Catalog,
                                          almacenamiento físico subyacente, scheduler del Job, parcheo y disponibilidad.

 Administra el EQUIPO (Grupo 30):        código de ingesta y transformación, diseño del esquema bronce/plata,
                                          definición de las dos tareas y sus dependencias en el Job, los GRANT y
                                          la matriz de roles, y las consultas de consumo sobre la capa plata.
```





In [0]:
import base64

def mostrar_imagen(ruta, ancho=900):
    with open(ruta, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    ext = ruta.split(".")[-1]
    displayHTML(f'<img src="data:image/{ext};base64,{b64}" style="max-width:{ancho}px;">')

mostrar_imagen("arquitectura_ea2.png")

### 3.2 Matriz de roles


| Rol | Bronce | Plata | Oro |
|---|---|---|---|
| Analista | Sin acceso/Solo consulta segun necesidad | Lectura (SELECT) | Lectura (SELECT) |
| Ingeniero de datos | Lectura y escritura | Lectura y escritura | Lectura y escritura |
| Administrador | Control total | Control total | Control total |

La matriz aplica al principio de mínimo privilegio. El profesional en análisis tiene principalmente permisos de lectura sobre los datos preparados para análisis, evitando así que se presenten modificaciones accidentales. El ingeniero de datos requiere permisos de lectura y escritura sobre las capas bronce, plata y oro porque es responsable de la ingesta y transformación de datos. El admin mantiene control sobre todas la capas para hacer gestiones de permisos, estructuras y operaciones administrativas. La capa gold se contempla como un capa de consumo de negocio, por lo que el analista tiene acceso de lectura 

### 3.3 Especificación del equivalente IaaS

Si en vez de Databricks (PaaS) el equipo tuviera que montar este mismo pipeline sobre máquinas virtuales puras (IaaS), esto es lo que habría que aprovisionar y mantener:

Máquinas y dimensionamiento
Dado el volumen actual (72,247 bookings, 100,000 clickstream, 99,793 reviews, 49,638 payments — unos cientos de miles de filas, no big data real), un clúster mínimo viable sería:

1 nodo driver: 4 vCPU / 16 GB RAM

2 nodos worker: 4 vCPU / 16 GB RAM cada uno

Esto permite paralelizar las transformaciones de la capa plata sin sobredimensionar para un volumen que hoy es modesto.

**Sistema operativo**

Ubuntu Server 22.04 LTS en cada nodo — es la base estándar para clústeres Spark autogestionados, con buen soporte de paquetes y actualizaciones de seguridad a largo plazo.

**Software a instalar**

Java (JDK 11 u OpenJDK 17, requerido por Spark)
Apache Spark en modo clúster (standalone o con YARN como gestor de recursos)
Delta Lake (librería, para conservar ACID/time travel que hoy da Unity Catalog de forma nativa)
Un metastore propio (Hive Metastore) para reemplazar el catálogo de Unity Catalog
Un orquestador externo (ej. Apache Airflow) para reemplazar el Job de Databricks y encadenar ingesta_bronce → transformar_plata
Un sistema de permisos propio (ej. Apache Ranger o políticas manuales sobre el metastore) para replicar la matriz de roles de la sección 3.2

**Red**

VPC con subred privada para los nodos worker (sin IP pública)
Subred pública solo para el nodo de acceso/bastión
Grupos de seguridad restringiendo puertos: SSH (22) solo desde IPs del equipo, puerto de la UI de Spark (8080/4040) accesible solo internamente
Balanceador o gateway si se expone algún endpoint de consulta hacia el equipo de marketing

**Almacenamiento**

Almacenamiento en bloque (disco) para el sistema operativo y Spark local de cada nodo
Un bucket de almacenamiento de objetos (equivalente a lo que hoy es el almacenamiento físico detrás de Unity Catalog) para persistir las tablas Delta de bronce y plata, con versionado activado para simular el time travel

**Estimación de esfuerzo**

Puesta en marcha: días, no minutos — instalar y configurar Spark, el metastore, la red y el orquestador antes de poder correr la primera consulta (frente a las horas que tomó tener el catálogo y las tablas plata funcionando en Databricks)
Operación: mantenimiento recurrente de parches de seguridad del SO, actualizaciones de versión de Spark, monitoreo manual de que el clúster esté sano y que el orquestador dispare el pipeline a tiempo — todo esto hoy lo absorbe el proveedor
### 3.4 Comparación IaaS / PaaS / SaaS

| Criterio | IaaS | PaaS | SaaS |
|---|---|---|---|
| Control |Total sobre SO, red y software instalado; el equipo configura todo desde cero |Control sobre código, datos y configuración de la plataforma, sin tocar el hardware subyacente |Mínimo; casi solo configuración de usuario final |
| Tiempo hasta el primer resultado |Semanas: aprovisionar VMs, instalar Spark/metastore/orquestador antes de correr una consulta |Horas: crear catálogo y empezar a correr notebooks de inmediato (lo que vivió el equipo) |Minutos: conectar el dato a una herramienta ya lista, sin flexibilidad de pipeline propio |
| Esfuerzo operativo |Alto y constante: parches, actualizaciones, monitoreo de infraestructura a cargo del equipo|Bajo: el proveedor gestiona clúster, motor de cómputo y metastore; el equipo solo mantiene código y permisos |Prácticamente nulo, no hay nada que administrar |
| Costo |Se paga por VM encendida, sin relación directa con el uso real |Se paga por cómputo consumido (alineado al uso real del pipeline) |Costo fijo por licencia/suscripción, sin relación con el volumen de datos |
| Escalabilidad |Manual: escalar implica aprovisionar más VMs y reconfigurar a mano |Elástica y administrada: el proveedor añade cómputo según la carga |Limitada a lo que la app del proveedor permita |
| Gobierno |El equipo diseña su propio esquema de permisos y auditoría desde cero | Ya integrado (Unity Catalog: GRANT, roles, linaje, como en 4.2-4.3)|Rara vez configurable; limitado a lo que la app expone |

**Conclusión:** para este caso conviene PaaS, que es justo lo que el equipo ya está usando. La razón central es que un equipo pequeño, con un plazo corto y sin experiencia previa administrando infraestructura, no puede permitirse el esfuerzo operativo y el tiempo de puesta en marcha que exige IaaS — y un SaaS, en el otro extremo, no nos daría el control de código y pipeline personalizado que la actividad exige (transformaciones propias, aplanado de structs, Jobs encadenados con dependencias). PaaS resuelve exactamente esa tensión: control suficiente sobre el código y los datos, sin tener que construir desde cero el gobierno de datos (GRANT, roles, linaje)

---
## 4. Implementación
### 4.1 Organización del entorno

In [0]:
#Organización entorno wanderbricks

CATALOGO = "bigdata_grupo30"
ESQUEMA = "wanderbricks"

tablas_bronce = [
    "bookings_bronce",
    "clickstream_bronce",
    "payments_bronce",
    "reviews_bronce"
]

tablas_plata = [
    "bookings_plata",
    "clickstream_plata",
    "payments_plata",
    "reviews_plata"
]

print(f"Catálogo: {CATALOGO}")
print(f"Esquema: {ESQUEMA}")

print("\nCapa Bronce:")
for tabla in tablas_bronce:
    print(f" - {CATALOGO}.{ESQUEMA}.{tabla}")

print("\nCapa Plata:")
for tabla in tablas_plata:
    print(f" - {CATALOGO}.{ESQUEMA}.{tabla}")

In [0]:
%sql
SHOW CATALOGS;

In [0]:
%sql
SHOW SCHEMAS IN bigdata_grupo30

In [0]:
%sql
SHOW TABLES IN bigdata_grupo30.wanderbricks;

Organizamos el entorno en dos capas dentro de un único esquema (bigdata_grupo30.wanderbricks), 
usando el sufijo de cada tabla para distinguir bronce de plata, en vez de crear un esquema separado 
por capa. Esto permite diferenciar permisos por capa (como se ve en 4.2) sin la complejidad adicional 
de gestionar tres esquemas distintos para un volumen de datos que no lo justifica.

### 4.2 Permisos


In [0]:
%sql
SELECT current_user();

In [0]:
%sql
SHOW GROUPS;

**Primer GRANT**

In [0]:
%sql
GRANT USE SCHEMA
ON SCHEMA bigdata_grupo30.wanderbricks
TO `account users`;

In [0]:
%sql
SHOW GRANTS ON SCHEMA bigdata_grupo30.wanderbricks;

Este primer GRANT le da a account users (el grupo que representa a todo el equipo en esta cuenta 
gratuita) el permiso mínimo de USE SCHEMA sobre wanderbricks. Sin este permiso, ningún otro GRANT 
más específico serviría de nada: es como dar acceso al edificio antes de dar la llave de una oficina 
puntual. Lo aplicamos a nivel de esquema, no de tabla, porque es un permiso que todo el equipo necesita 
sin excepción.

**Segundo GRANT**

In [0]:
%sql
GRANT SELECT
ON TABLE bigdata_grupo30.wanderbricks.bookings_plata
TO `account users`;

In [0]:
%sql
SHOW GRANTS ON TABLE bigdata_grupo30.wanderbricks.bookings_plata;

El segundo GRANT es más restrictivo a propósito: SELECT solo sobre bookings_plata, no sobre toda la 
capa. Simula el permiso que tendría un analista de negocio (ver la tabla 3.2): puede consultar 
resultados ya limpios, pero no puede escribir ni ver las tablas de bronce sin procesar. La diferencia 
de nivel entre los dos GRANT (USE SCHEMA a nivel general vs SELECT sobre una tabla específica) es 
justo lo que pide la rúbrica: "niveles distintos sobre objetos distintos".

### 4.3 Linaje

Capturamos el linaje de clickstream_plata desde el Catalog Explorer. El grafo muestra la cadena completa: samples.wanderbricks.clickstream → clickstream_bronce → clickstream_plata. Se puede ver además cómo cambia la estructura en el camino: clickstream_bronce conserva la columna metadata como struct sin abrir, mientras que clickstream_plata ya la tiene desagregada en device y referrer como columnas normales, evidencia visual de la transformación aplicada en la capa plata.



In [0]:
import base64

def mostrar_imagen(ruta, ancho=900):
    with open(ruta, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    ext = ruta.split(".")[-1]
    displayHTML(f'<img src="data:image/{ext};base64,{b64}" style="max-width:{ancho}px;">')

mostrar_imagen("linaje_clickstream.jpg")

### 4.4 Automatización



In [0]:
# ── Tarea 1: ingesta_bronce ──
from pyspark.sql import functions as F

CATALOGO = "bigdata_grupo30"
ESQUEMA = "wanderbricks"

tablas_negocio = ["bookings", "clickstream", "reviews", "payments"]

for tabla in tablas_negocio:
    tabla_bronce = f"{CATALOGO}.{ESQUEMA}.{tabla}_bronce"
    df = (spark.table(f"samples.wanderbricks.{tabla}")
            .withColumn("_ingesta_ts", F.current_timestamp())
            .withColumn("_origen", F.lit(f"samples.wanderbricks.{tabla}")))
    (df.write.format("delta").mode("overwrite")
       .option("overwriteSchema", "true").saveAsTable(tabla_bronce))
    print(f"✅ {tabla_bronce}: {df.count():,} registros")


# ── Tarea 2: transformar_plata (depende de que la Tarea 1 termine bien) ──
bookings_plata = (spark.table(f"{CATALOGO}.{ESQUEMA}.bookings_bronce")
    .dropDuplicates(["booking_id"])
    .filter(F.col("total_amount").isNotNull())
    .withColumn("total_amount", F.col("total_amount").cast("decimal(10,2)")))
bookings_plata.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOGO}.{ESQUEMA}.bookings_plata")

payments_plata = (spark.table(f"{CATALOGO}.{ESQUEMA}.payments_bronce")
    .dropDuplicates(["payment_id"])
    .filter(F.col("amount").isNotNull()))
payments_plata.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOGO}.{ESQUEMA}.payments_plata")

reviews_plata = (spark.table(f"{CATALOGO}.{ESQUEMA}.reviews_bronce")
    .filter(F.col("is_deleted") == False)
    .dropDuplicates(["review_id"]))
reviews_plata.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOGO}.{ESQUEMA}.reviews_plata")

clickstream_plata = (spark.table(f"{CATALOGO}.{ESQUEMA}.clickstream_bronce")
    .withColumn("device", F.col("metadata.device"))
    .withColumn("referrer", F.col("metadata.referrer"))
    .drop("metadata"))
clickstream_plata.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOGO}.{ESQUEMA}.clickstream_plata")

print("✅ Capa plata reconstruida: bookings, payments, reviews, clickstream")

Armamos un Job (pipeline_wanderbricks, Job ID 680860470795814) con dos tareas encadenadas: ingesta_bronce, que trae las 4 tablas desde samples.wanderbricks a la capa bronce, y transformar_plata, que solo arranca si la primera terminó exitosamente (configurado con "Depends on: ingesta_bronce" y "Run if dependencies: All succeeded"). Se programó con una frecuencia diaria y se desactivó inmediatamente después de capturar la evidencia de una ejecución exitosa, para no consumir la cuota del plan gratuito.



In [0]:
import base64

def mostrar_imagen(ruta, ancho=900):
    with open(ruta, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    ext = ruta.split(".")[-1]
    displayHTML(f'<img src="data:image/{ext};base64,{b64}" style="max-width:{ancho}px;">')

mostrar_imagen("pipeline_job.jpg")

---
## 5. Resultados

El diagrama de arquitectura (3.1) delimita con claridad la frontera de responsabilidad: Databricks 
administra el cómputo, el motor Spark, el metastore de Unity Catalog y el almacenamiento físico, 
mientras que el equipo diseñó el esquema bronce/plata, el código de transformación y las dos tareas 
encadenadas del Job. Esa misma frontera se hizo tangible al comparar con el equivalente IaaS (3.3): 
levantar esto mismo sobre máquinas virtuales habría tomado días de instalación de Spark, metastore y 
orquestador antes de correr una sola consulta, frente a las horas que tomó tener el catálogo 
funcionando en Databricks lo que sustenta la conclusión de la comparación en 3.4, que PaaS es la 
opción que mejor balancea control y esfuerzo operativo para un equipo de tres personas con un plazo 
corto.

Sobre esa arquitectura, la organización del entorno (4.1) quedó verificable con SHOW TABLES: 8 tablas 
en el esquema wanderbricks, 4 en bronce y 4 en plata, siguiendo la matriz de roles diseñada en 3.2. 
Los dos GRANT de niveles distintos (4.2) se confirmaron con SHOW GRANTS: account users obtuvo USE 
SCHEMA sobre todo el esquema wanderbricks (el permiso base, sin el cual ningún otro GRANT serviría) 
y, por separado, SELECT únicamente sobre bookings_plata (el permiso acotado que tendría un analista 
de negocio, sin acceso a los datos crudos de bronce).

Por último, el linaje de clickstream_plata (4.3) mostró la cadena completa desde 
samples.wanderbricks.clickstream hasta la tabla final, evidenciando además el aplanado del struct 
metadata en device y referrer. Y el Job pipeline_wanderbricks (4.4) ejecutó sus dos tareas 
encadenadas: ingesta_bronce y transformar_plata, con "Depends on" y "Run if dependencies: All 
succeeded"  reproduciendo exactamente los mismos conteos que la EA1 (72,247 bookings, 100,000 
clickstream, 99,793 reviews, 49,638 payments), lo que confirma que automatizar el proceso no alteró 
en nada los datos originales.

---
## 6. Conclusiones

Buscábamos que el entorno de Wanderbricks pasara de ser una base de datos donde cualquiera podía ver 
y modificar todo, a un entorno con acceso diferenciado y un pipeline que se mantuviera solo. En ese 
sentido, el resultado fue el esperado: el Job encadenado reprodujo exactamente los mismos conteos que 
obtuvimos manualmente en la EA1 (72,247 bookings, 100,000 clickstream, 99,793 reviews, 49,638 
payments), lo que confirma que automatizar el proceso no alteró en nada los datos, solo quitó la 
necesidad de que alguien le diera clic.

Lo que no anticipábamos fue que el mayor obstáculo no sería técnico sino de coordinación entre cuentas. 
Databricks Free Edition le da a cada integrante su propio espacio de trabajo, así que un GRANT dirigido 
al correo de una persona específica solo funciona si se ejecuta desde su propia cuenta: cuando lo 
corrimos desde otra, Unity Catalog simplemente no reconocía a ese usuario como válido. Tuvimos que 
cambiar los permisos hacia el grupo account users, algo que en un entorno corporativo real con grupos 
propios por rol no habría sido necesario. Y editar el mismo notebook desde Git en paralelo generó 
varios conflictos de fusión, algunos con marcadores de conflicto que llegaron a quedar confirmados en 
la rama principal y rompieron el archivo por completo; resolverlos nos tomó bastante más tiempo del 
que había tomado escribir el código original.

Si repitiéramos este trabajo, cambiaríamos dos cosas. Primero, definiríamos desde el inicio que una 
sola persona consolida y sube el notebook compartido, mientras el resto aporta su código y capturas 
por separado evitaría por completo los conflictos de fusión que enfrentamos. Segundo, usaríamos 
grupos genéricos en los permisos desde la primera versión, en lugar de correos individuales, ya que 
es lo que de verdad se sostiene quien accede desde una cuenta distinta a la que lo configuró.

---
## 7. Reparto del trabajo y uso de IA

| Integrante | De qué se encargó | Qué sustenta en el video |
|---|---|---|
| Andres Felipe Grisales Vahos | Parte escrita: diagrama de arquitectura (3.1), especificación del equivalente IaaS (3.3) y comparación IaaS/PaaS/SaaS con conclusión (3.4). | Pregunta 1: qué administra el proveedor y qué administra el equipo. |
| Hanna Jineth Contreras Salinas | Linaje (4.3) y automatización con Jobs (4.4), además de la Sección 1, 2 y la organización final del documento. | Pregunta 3: qué se complicaría montando esto sobre máquinas virtuales. |
| Arley Steven Franco Galeano | Gobierno y permisos: organización del entorno con criterio (4.1), los 2 GRANT con SHOW GRANTS y la matriz de roles (3.2, 4.2). | Pregunta 2: un GRANT que ejecutó, a quién y por qué. |

**Uso de asistentes de IA:** Usamos NotebookLM (Google) como apoyo para el estudio, subiendo el laboratorio guiado y los notebooks 
del curso para generar mapas conceptuales que nos ayudaron a entender mejor el gobierno de datos, el 
linaje y la automatización con Jobs antes de aplicarlos, igual que hicimos en la EA1. Además usamos 
Claude (Anthropic) puntualmente para diagnosticar y resolver un problema técnico: el notebook 
compartido quedó roto varias veces por conflictos de fusión de Git sin resolver, con marcadores de 
conflicto que dañaban el archivo JSON, y Claude ayudó a identificar la causa y a reconstruir el 
notebook sin perder el trabajo de nadie. El código y las decisiones de diseño fueron discutidos y 
entendidos por el grupo antes de incluirlos; cada integrante puede explicar cualquier línea de la 
parte que le correspondió.